# NavGo — Colab serve + Cloudflare Tunnel

**TR:** Eğitilmiş Gemma + LoRA’yı bu Colab GPU’sunda çalıştırır ve Cloudflare ile dışarı açar.

**EN:** Loads Gemma + NavGo LoRA on this Colab GPU and publishes an HTTPS URL via Cloudflare.

Runtime → Change runtime type → **T4 GPU**. Keep the last cell running; closing the tab kills the tunnel.

Docs: `deployments/cloudflare/CLOUDFLARE.md` in the NavGo repo.


## 1. Config


In [ ]:
import os, secrets, sys
from pathlib import Path

# Optional: clone NavGo if server/ is not already on the runtime.
GH_TOKEN = os.environ.get("GH_TOKEN", "").strip()  # only if repo is private
REPO_URL = os.environ.get("NAVGO_REPO_URL", "https://github.com/leventkok/NavGo.git")
# Until cloudflare lands on main, set NAVGO_REPO_BRANCH to your push branch.
REPO_BRANCH = os.environ.get("NAVGO_REPO_BRANCH", "feat/web-gemma-and-license")
REPO_DIR = Path("/content/NavGo")
SERVER_DIR = Path("/content/navgo-llm/server")

BASE_MODEL = os.environ.get("NAVGO_BASE_MODEL", "google/gemma-2-2b-it")
ADAPTER_ID = os.environ.get("NAVGO_ADAPTER_ID", "levonov/navgo-gemma-lora-v3")  # "" = base only
SERVED_MODEL = "navgo-gemma"
LOAD_IN_4BIT = True

# Must match Render / masterfabric-go/.env LLM_API_KEY. Leave empty to auto-generate.
LLM_API_KEY = os.environ.get("LLM_API_KEY", "").strip()
PORT = 8000

# Optional named Cloudflare tunnel. Empty = rotating trycloudflare.com URL.
CLOUDFLARE_TUNNEL_TOKEN = os.environ.get("CLOUDFLARE_TUNNEL_TOKEN", "").strip()
LLM_PUBLIC_URL = os.environ.get("LLM_PUBLIC_URL", "").strip()

# Required for gated Gemma: https://huggingface.co/settings/tokens
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

if not LLM_API_KEY:
    LLM_API_KEY = secrets.token_urlsafe(24)
os.environ["LLM_API_KEY"] = LLM_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("API key ready:", LLM_API_KEY[:6] + "…" + LLM_API_KEY[-4:])
print("base:", BASE_MODEL)
print("adapter:", ADAPTER_ID or "(none)")
print("HF_TOKEN set:", bool(HF_TOKEN))


## 2. GPU check + fetch server code


In [ ]:
import shutil
import subprocess
from pathlib import Path

!nvidia-smi -L

marker = SERVER_DIR / "app.py"
if not marker.exists():
    # Prefer uploading deployments/cloudflare/server to /content/navgo-llm/server
    uploaded = Path("/content/server")
    if (uploaded / "app.py").exists():
        SERVER_DIR.parent.mkdir(parents=True, exist_ok=True)
        if SERVER_DIR.exists():
            shutil.rmtree(SERVER_DIR)
        shutil.copytree(uploaded, SERVER_DIR)
    else:
        if not (REPO_DIR / "masterfabric-go" / "deployments" / "cloudflare" / "server" / "app.py").exists():
            clone_url = REPO_URL
            if GH_TOKEN:
                clone_url = REPO_URL.replace("https://", f"https://x-access-token:{GH_TOKEN}@")
            if REPO_DIR.exists():
                shutil.rmtree(REPO_DIR)
            subprocess.check_call(
                ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)],
            )
        src = REPO_DIR / "masterfabric-go" / "deployments" / "cloudflare" / "server"
        if not (src / "app.py").exists():
            raise SystemExit(
                "server/app.py missing. Push deployments/cloudflare to the repo, "
                "or upload server/ to /content/server"
            )
        SERVER_DIR.parent.mkdir(parents=True, exist_ok=True)
        if SERVER_DIR.exists():
            shutil.rmtree(SERVER_DIR)
        shutil.copytree(src, SERVER_DIR)

sys.path.insert(0, str(SERVER_DIR))
print("server:", marker.exists(), SERVER_DIR)


## 3. Adapter path check (Drive only if local path)


In [ ]:
from pathlib import Path

adapter = Path(ADAPTER_ID) if ADAPTER_ID and ADAPTER_ID.startswith("/") else None
if adapter and str(adapter).startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("adapter exists:", adapter.exists(), adapter)
elif adapter:
    print("adapter path:", adapter, "exists:", adapter.exists())
else:
    print("adapter id (HF or empty):", ADAPTER_ID or "(base only)")


## 4. Install serving deps


In [ ]:
%pip install -q fastapi uvicorn pydantic transformers peft bitsandbytes accelerate sentencepiece
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login ok")
else:
    print("HF_TOKEN empty — gated models (Gemma) will fail to download")


## 5. Load model


In [ ]:
from engine import TransformersEngine
from app import create_app, ensure_api_key

ensure_api_key()
engine = TransformersEngine(
    BASE_MODEL,
    ADAPTER_ID or None,
    load_in_4bit=LOAD_IN_4BIT,
    served_model=SERVED_MODEL,
    hf_token=HF_TOKEN or None,
)
engine.load()
print("loaded:", engine.model_id(), "ready:", engine.ready())


## 6. Start FastAPI + Cloudflare tunnel

This cell blocks. Copy the printed `LLM_BASE_URL` / `LLM_API_KEY` into Render env or `masterfabric-go/.env`.


In [ ]:
import threading, time, uvicorn
from tunnel import download_cloudflared, start_tunnel, wait_for_health, write_env_snippet

api_app = create_app(engine, api_key=LLM_API_KEY)

def _run():
    uvicorn.run(api_app, host="0.0.0.0", port=PORT, log_level="info")

t = threading.Thread(target=_run, daemon=True)
t.start()
local = f"http://127.0.0.1:{PORT}"
assert wait_for_health(local, timeout_s=60), "local /health did not come up"

os.environ["CLOUDFLARE_TUNNEL_TOKEN"] = CLOUDFLARE_TUNNEL_TOKEN
if LLM_PUBLIC_URL:
    os.environ["LLM_PUBLIC_URL"] = LLM_PUBLIC_URL

binary = download_cloudflared()
proc, public_url = start_tunnel(local, binary=binary, token=CLOUDFLARE_TUNNEL_TOKEN or None)
if not public_url:
    raise RuntimeError(
        "cloudflared started but no public URL. "
        "For named tunnels set LLM_PUBLIC_URL; for quick tunnels check the process log."
    )

print("\n===== paste into Render / .env =====")
print(write_env_snippet(public_url, LLM_API_KEY))
print("===================================\n")
print("Probe:")
print(f"  curl -s {public_url}/health")
print(
    f"  curl -s -o /dev/null -w '%{{http_code}}\\n' -X POST {public_url}/v1/chat/completions "
    f"-H 'Content-Type: application/json' -d '{{}}'"
)
print("  (second command must be 401 without a key)")

while proc.poll() is None:
    time.sleep(30)
raise RuntimeError(f"cloudflared exited with {proc.returncode}")
